In [0]:


%sql
create schema if not exists yelp_dataset.silver;

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
import logging

logger = logging.getLogger("SilverETL")
logger.setLevel(logging.INFO)
class SilverETL:
    def __init__(self):
        self.spark = spark
        self.catalog = "yelp_dataset"
        self.bronze_schema = "bronze"
        self.silver_schema = "silver"
    def load_bronze_data(self, table_name: str)-> DataFrame:

        bronze_table = f"{self.catalog}.{self.bronze_schema}.{table_name}"

        try:
            if not self.spark.catalog.tableExists(bronze_table):
                logger.warning(f"Bronze table {bronze_table} not found")
                return None
            df = self.spark.table(bronze_table)
            logger.info(f"loaded {df.count()} records from bronze {table_name}")
            return df
        except Exception as e:
            logger.error(f"Failed to load {bronze_table}: {e}")
            return None
    def clean_business_data(self, df :DataFrame) -> DataFrame:
        logger.info("Cleaning Business Data")
        df = df.dropDuplicates(["business_id"])
        df= df.withColumn("business_id",trim(col("business_id")))
        df = df.withColumn("name", trim(col('name')))\
                .withColumn("address", trim(col('address')))\
                .withColumn("city", trim(col('city')))\
                .withColumn("state", trim(col('state')))\
                .withColumn("postal_code", regexp_replace(trim(col('postal_code')),r"[^0-9-]",""))
        df=df.withColumn("latitude",col("latitude").cast("double")) \
              .withColumn("longitude",col("longitude").cast("double")) \
              .withColumn("stars",col("stars").cast("double")) \
              .withColumn("review_count",col("review_count").cast("long")) \
              .withColumn("is_open",col("is_open").cast("integer"))
        df=df.fillna({
            "address":"Unknown",
            "city":"Unknown",
            "state":"Unknown",
            "postal_code":"00000",
            "stars":0.0,
            "review_count":0,
            "is_open":0
        })
        df=df.withColumn("categories_array",when(col("categories").isNotNull() & (trim(col("categories"))!=""),split(col("categories"),", ")).otherwise(array()))
        df=df.withColumn("is_restaurant",when(array_contains(col("categories_array"),"Restaurants"),True).otherwise(False))
        df=df.withColumn("has_location",when(col("latitude").isNotNull() & col("longitude").isNotNull(),True).otherwise(False))
        df=df.withColumn("valid_latitude",when(col("latitude").between(-90,90),True).otherwise(False))
        df=df.withColumn("valid_longitude",when(col("longitude").between(-180,180),True).otherwise(False))
        df=df.withColumn("valid_stars",when(col("stars").between(1,5),True).otherwise(False))
        df=df.withColumn("valid_review_count",when(col("review_count")>=0,True).otherwise(False))
        df=df.withColumn("has_reviews",when(col("review_count")>0,True).otherwise(False))
        df=df.withColumn("valid_business_id",when(col("business_id").isNotNull() & (trim(col("business_id"))!=""),True).otherwise(False))
        
        # Enhanced transformations for better analytics
        # 1. Extract key attributes from nested struct
        df = df.withColumn("price_range", col("attributes.RestaurantsPriceRange2")) \
               .withColumn("takes_reservations", col("attributes.RestaurantsReservations")) \
               .withColumn("outdoor_seating", col("attributes.OutdoorSeating")) \
               .withColumn("alcohol", col("attributes.Alcohol")) \
               .withColumn("good_for_kids", col("attributes.GoodForKids")) \
               .withColumn("parking_available", 
                          when(col("attributes.BusinessParking").isNotNull(), True).otherwise(False))
        
        # 2. Calculate popularity score (weighted: 60% stars, 40% review volume)
        # df = df.withColumn("popularity_score", 
        #                   ((col("stars")/5.0) * 0.6 + 
        #                    (least(col("review_count"), lit(1000)) / 1000.0) * 40).cast("double"))
        
        # 3. Business maturity indicator based on review count
        df = df.withColumn("business_maturity", 
                          when(col("review_count") >= 500, "established")
                          .when(col("review_count") >= 100, "growing")
                          .when(col("review_count") >= 20, "emerging")
                          .otherwise("new"))
        
        # 4. Rating tier classification
        df = df.withColumn("rating_tier", 
                          when(col("stars") >= 4.5, "excellent")
                          .when(col("stars") >= 4.0, "very_good")
                          .when(col("stars") >= 3.5, "good")
                          .when(col("stars") >= 3.0, "average")
                          .otherwise("below_average"))
        
        # 5. Standardize geographic data for consistent grouping
        df = df.withColumn("city_normalized", upper(trim(col("city")))) \
               .withColumn("state_normalized", upper(trim(col("state"))))
        
        # 6. Category flags for common business types
        df = df.withColumn("is_fast_food", array_contains(col("categories_array"), "Fast Food")) \
               .withColumn("is_bar", array_contains(col("categories_array"), "Bars")) \
               .withColumn("is_cafe", array_contains(col("categories_array"), "Coffee & Tea")) \
               .withColumn("is_nightlife", array_contains(col("categories_array"), "Nightlife")) \
               .withColumn("category_count", size(col("categories_array")))
        #7. Hours category for business operating hours
        df = df.withColumn("monday_hours",col("hours.Monday")) \
                .withColumn("tuesday_hours",col("hours.Tuesday")) \
                .withColumn("wednesday_hours",col("hours.Wednesday")) \
                .withColumn("thursday_hours",col("hours.Thursday")) \
                .withColumn("friday_hours",col("hours.Friday")) \
                .withColumn("saturday_hours",col("hours.Saturday")) \
                .withColumn("sunday_hours",col("hours.Sunday"))\
                .withColumn("days_open_count", (when(col("monday_hours").isNotNull(),1).otherwise(0)+
                when(col("tuesday_hours").isNotNull(),1).otherwise(0)+
                 when(col("wednesday_hours").isNotNull(),1).otherwise(0)+
                 when(col("thursday_hours").isNotNull(),1).otherwise(0)+
                 when(col("friday_hours").isNotNull(),1).otherwise(0)+
                 when(col("saturday_hours").isNotNull(),1).otherwise(0)+
                 when(col("sunday_hours").isNotNull(),1).otherwise(0)
                            ))


        df=df.withColumn("processed_timestamp",current_timestamp()) \
              .withColumn("data_layer",lit("silver"))
        logger.info(f"Business data cleaned: {df.count()} records")
        return df   

    def clean_review_data(self,df:DataFrame) -> DataFrame:
        
        logger.info("Cleaning Review data")
        df = df.dropDuplicates(["review_id"])
        df = df.withColumn("review_id", trim(col('review_id')))\
                .withColumn("business_id", trim(col('business_id')))\
                .withColumn("user_id", trim(col('user_id')))
        df = df.withColumn("text_cleaned", regexp_replace(col("text"), r"[^\w\s.,!?-]",""))
        df = df.withColumn("text_length", length(coalesce(col("text"), lit(""))))
        df = df.withColumn("review_date", to_date(col("date"), "yyyy-MM-dd HH:mm:ss"))
        fill_values ={}
        if "useful" in df.columns:
             fill_values["useful"]=0
        if "funny" in df.columns:
            fill_values["funny"]=0
        if "cool" in df.columns:
            fill_values["cool"]=0
        if "stars" in df.columns:
            fill_values["stars"]=0
        if fill_values:
            df= df.fillna(fill_values)
        df=df.withColumn("stars",col("stars").cast("double"))
        df=df.withColumn("valid_review_id",when(col("review_id").isNotNull() & (trim(col("review_id"))!=""), True).otherwise(False))
        df=df.withColumn("valid_business_id",when(col("business_id").isNotNull() & (trim(col("business_id"))!=""), True).otherwise(False))
        df=df.withColumn("valid_user_id",when(col("user_id").isNotNull() & (trim(col("user_id"))!=""),True).otherwise(False))
        df=df.withColumn("valid_stars",when(col("stars").between(1,5),True).otherwise(False))
        df=df.withColumn("valid_review_date",when(col("review_date").isNotNull(),True).otherwise(False))
        df=df.withColumn("has_text",when(col("text_length")>10,True).otherwise(False))
        df=df.withColumn("review_quality",when(col("text_length")>100,"high").when(col("text_length")>50,"medium").otherwise("low"))
        df=df.withColumn("processed_timestamp",current_timestamp()) \
              .withColumn("data_layer",lit("silver"))
        logger.info(f"Review data closed: {df.count()} records")
        return df
    def clean_user_data(self, df:DataFrame)->DataFrame:
        logger.info("Cleaning user data")
        df = df.dropDuplicates(["user_id"])
        df = df.withColumn("user_id",trim(col('user_id')))\
                .withColumn("name", trim(col("name")))
        df = df.withColumn("yelp_since_date", to_date(col("yelping_since"), "yyyy-MM-dd HH:mm:ss"))
        df = df.withColumn("user_tenure_days", datediff(current_date(),col("yelp_since_date")))
        numerical_cols = ["review_count", "useful", "funny", "cool", "fans", "average_stars"]
        for col_name in numerical_cols:
            if col_name in df.columns:
                df=df.fillna({col_name:0})
        for col_name in ["review_count","useful","funny","cool","fans"]:
            if col_name in df.columns:
                df=df.withColumn(col_name,col(col_name).cast("long"))
        if "average_stars" in df.columns:
            df=df.withColumn("average_stars",col("average_stars").cast("double"))
        df=df.withColumn("elite_years",when(col("elite").isNotNull() & (trim(col("elite"))!=""),split(col("elite"),",")).otherwise(array()))
        df=df.withColumn("elite_years_count",size(col("elite_years")))
        df=df.withColumn("is_elite",when(col("elite_years_count")>0,True).otherwise(False))
        df=df.withColumn("friends_array",when(col("friends").isNotNull() & (trim(col("friends"))!=""),split(col("friends"),", ")).otherwise(array()))
        df=df.withColumn("friends_count",size(col("friends_array")))
        df=df.withColumn("valid_user_id",when(col("user_id").isNotNull() & (trim(col("user_id"))!=""),True).otherwise(False))
        df=df.withColumn("valid_review_count",when(col("review_count")>=0,True).otherwise(False))
        df=df.withColumn("valid_useful",when(col("useful")>=0,True).otherwise(False))
        df=df.withColumn("valid_funny",when(col("funny")>=0,True).otherwise(False))
        df=df.withColumn("valid_cool",when(col("cool")>=0,True).otherwise(False))
        df=df.withColumn("valid_fans",when(col("fans")>=0,True).otherwise(False))
        df=df.withColumn("valid_average_stars",when(col("average_stars").between(1,5),True).otherwise(False))
        df=df.withColumn("engagement_score",(col("review_count")*0.4+col("useful")*0.2+col("funny")*0.1+col("cool")*0.1+col("fans")*0.2))
        df=df.withColumn("user_tier",when(col("review_count")>=100,"power_user").when(col("review_count")>=50,"active_user").when(col("review_count")>=10,"regular_user").otherwise("casual_user"))
        df=df.withColumn("processed_timestamp",current_timestamp()) \
              .withColumn("data_layer",lit("silver"))
        logger.info(f"User data cleaned: {df.count()} records")
        return df
    def clean_checkin_data(self, df:DataFrame)-> DataFrame:
        logger.info("Cleaning Checkin Data")
        df = df.dropDuplicates(['business_id'])
        df = df.withColumn("business_id", trim(col('business_id')))
        df=df.withColumn("checkin_dates_array",when(col("date").isNotNull() & (trim(col("date"))!=""),split(col("date"),", ")).otherwise(array()))
        df=df.withColumn("checkin_count",size(col("checkin_dates_array")))
        df=df.withColumn("checkin_timestamp_array",transform(col("checkin_dates_array"),lambda x:to_timestamp(x,"yyyy-MM-dd HH:mm:ss")))
        df = df.withColumn("first_checkin", array_min(col("checkin_timestamp_array")))
        df = df.withColumn("last_checkin", array_max(col("checkin_timestamp_array")))
        df=df.withColumn("valid_business_id",when(col("business_id").isNotNull() & (trim(col("business_id"))!=""),True).otherwise(False))
        df=df.withColumn("has_checkins",when(col("checkin_count")>0,True).otherwise(False))
        df= df.withColumn("processed_timestamp", current_timestamp())\
            .withColumn("data_layer", lit("silver"))
        df= df.withColumn("business_age_days", datediff(col("last_checkin"), col("first_checkin")))\
            .withColumn("business_age_years", (col("business_age_days")/365.0).cast("double"))
        logger.info(f"Checkin data cleaned: {df.count()} records")
        return df
    def clean_tip_data(self, df: DataFrame)-> DataFrame:
        logger.info("Cleaning tip Data")
        df = df.dropDuplicates(['user_id','text','date','business_id'])
        df =df.withColumn("user_id",trim(col("user_id")))\
            .withColumn("business_id",trim(col("business_id")))
        df=df.withColumn("text_cleaned",regexp_replace(col("text"),r"[^\w\s.,!?-]",""))
        df=df.withColumn("tip_date",to_date(col('date'),"yyyy-MM-dd HH:mm:ss"))
        df =df.withColumn("text_length", length(coalesce(col("text"), lit(""))))
        if "compliment_count" in df.columns:
            df=df.fillna({"compliment_count":0})
            df=df.withColumn("compliment_count",col("compliment_count").cast("long"))
        df=df.withColumn("valid_user_id",when(col("user_id").isNotNull() & (trim(col("user_id"))!=""),True).otherwise(False))
        df=df.withColumn("valid_business_id", when(col("business_id").isNotNull()& (trim(col("business_id"))!=""),True).otherwise(False))
        df=df.withColumn("valid_tip_date", when(col("tip_date").isNotNull(),True).otherwise(False))
        df=df.withColumn("has_text", when(col("text_length")>0,True).otherwise(False))
        df=df.withColumn("processed_timestamp",current_timestamp()) \
              .withColumn("data_layer",lit("silver"))
        logger.info(f"Tip data cleaned: {df.count()} records")
        return df
    def save_to_silver(self, df: DataFrame, table_name: str):
        """Save DataFrame to silver layer"""
        silver_table=f"{self.catalog}.{self.silver_schema}.{table_name}"
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table)
        logger.info(f"Saved {silver_table} to silver layer")

    def generate_data_quality_report(self):
        """Generate data quality report for silver layer"""
        logger.info("Generating data quality report")
        quality_metrics=[]
        tables=["yelp_business","yelp_review","yelp_user","yelp_checkin","yelp_tip"]
        key_columns={
            "yelp_business":["business_id","name","city","state","stars","review_count","latitude","longitude"],
            "yelp_review":["review_id","business_id","user_id","stars","review_date","text"],
            "yelp_user":["user_id","name","yelping_since_date","review_count","average_stars"],
            "yelp_checkin":["business_id","checkin_count"],
            "yelp_tip":["user_id","business_id","tip_date","text"]
        }
        for table_name in tables:
            silver_table=f"{self.catalog}.{self.silver_schema}.{table_name}"
            if not self.spark.catalog.tableExists(silver_table):
                continue
            df=self.spark.table(silver_table)
            total_count=df.count()
            metrics={"table_name":table_name,"total_records":total_count}
            for col_name in key_columns[table_name]:
                if col_name in df.columns:
                    null_count=df.filter(col(col_name).isNull()).count()
                    metrics[f"{col_name}_null_percentage"]=(null_count/total_count*100) if total_count>0 else 0
            quality_metrics.append(metrics)
        if quality_metrics:
            report_df=self.spark.createDataFrame(quality_metrics)
            report_df=report_df.withColumn("report_timestamp",current_timestamp())
            report_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{self.catalog}.{self.silver_schema}.data_quality_report")
            logger.info("Data quality report saved to yelp_dataset.silver.data_quality_report")
            return report_df
        return None
    def run(self)->bool:
        """Run complete silver layer ETL"""
        logger.info("Starting Silver Layer ETL")
        datasets={}
        business_df=self.load_bronze_data("yelp_business")
        if business_df is not None:
            business_clean=self.clean_business_data(business_df)
            self.save_to_silver(business_clean,"yelp_business")
            datasets["business"]=business_clean
        review_df=self.load_bronze_data("yelp_review")
        if review_df is not None:
            review_clean=self.clean_review_data(review_df)
            self.save_to_silver(review_clean,"yelp_review")
            datasets["review"]=review_clean
        user_df=self.load_bronze_data("yelp_user")
        if user_df is not None:
            user_clean=self.clean_user_data(user_df)
            self.save_to_silver(user_clean,"yelp_user")
            datasets["user"]=user_clean
        checkin_df=self.load_bronze_data("yelp_checkin")
        if checkin_df is not None:
            checkin_clean=self.clean_checkin_data(checkin_df)
            self.save_to_silver(checkin_clean,"yelp_checkin")
            datasets["checkin"]=checkin_clean
        tip_df=self.load_bronze_data("yelp_tip")
        if tip_df is not None:
            tip_clean=self.clean_tip_data(tip_df)
            self.save_to_silver(tip_clean,"yelp_tip")
            datasets["tip"]=tip_clean
        self.generate_data_quality_report()
        logger.info("Silver Layer ETL completed successfully")
        return True
silver_etl=SilverETL()
silver_etl.run()

In [0]:
# # Initialize the SilverETL class
# silver_etl = SilverETL()

# # Load and clean business data
# # print("=" * 80)
# # print("BUSINESS DATA")
# # print("=" * 80)
# # business_bronze = silver_etl.load_bronze_data("business")
# # if business_bronze:
# #     business_silver = silver_etl.clean_business_table(business_bronze)
# #     print(f"\nCleaned business records: {business_silver.count()}")
# #     print("\nSample business data:")
# #     display(business_silver.limit(5))
# # else:
# #     print("No business data found in bronze layer")

# # Load and clean review data
# # print("\n" + "=" * 80)
# # print("REVIEW DATA")
# # print("=" * 80)
# # review_bronze = silver_etl.load_bronze_data("yelp_review")
# # if review_bronze:
# #     review_silver = silver_etl.clean_review_data(review_bronze)
# #     print(f"\nCleaned review records: {review_silver.count()}")
# #     print("\nSample review data:")
# #     display(review_silver.limit(5))
# # else:
# #     print("No review data found in bronze layer")

# # Load and clean user data
# # print("\n" + "=" * 80)
# # print("USER DATA")
# # print("=" * 80)
# # user_bronze = silver_etl.load_bronze_data("yelp_user")
# # if user_bronze:
# #     user_silver = silver_etl.clean_user_data(user_bronze)
# #     print(f"\nCleaned user records: {user_silver.count()}")
# #     print("\nSample user data:")
# #     display(user_silver.limit(5))
# # else:
# #     print("No user data found in bronze layer")

# # Load and clean checkin data
# # print("\n" + "=" * 80)
# # print("CHECKIN DATA")
# # print("=" * 80)
# # checkin_bronze = silver_etl.load_bronze_data("yelp_checkin")
# # if checkin_bronze:
# #     checkin_silver = silver_etl.clean_checkin_data(checkin_bronze)
# #     print(f"\nCleaned checkin records: {checkin_silver.count()}")
# #     print("\nSample checkin data:")
# #     display(checkin_silver.limit(5))
# # else:
# #     print("No checkin data found in bronze layer")

# # Load and clean tip data
# print("\n" + "=" * 80)
# print("TIP DATA")
# print("=" * 80)
# tip_bronze = silver_etl.load_bronze_data("yelp_tip")
# if tip_bronze:
#     tip_silver = silver_etl.clean_tip_data(tip_bronze)
#     print(f"\nCleaned tip records: {tip_silver.count()}")
#     print("\nSample tip data:")
#     display(tip_silver.limit(5))
# else:
#     print("No tip data found in bronze layer")

In [0]:
for table in ["yelp_business","yelp_review","yelp_user","yelp_checkin","yelp_tip"]:
    df=spark.table(f"yelp_dataset.silver.{table}")
    print(f"{table}: {df.count()}")

In [0]:
spark.table("yelp_dataset.silver.yelp_business").printSchema()

In [0]:
spark.table("yelp_dataset.silver.yelp_review").printSchema()